In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import os
import time
import pandas as pd

# --- WebDriver Setup ---
chromedriver_path = os.path.join(os.getcwd(), "chromedriver")

service = Service(executable_path=chromedriver_path)

game_id = 2651280
url_template = "https://steamcommunity.com/app/{}/reviews/?p=1&browsefilter=mostrecent&filterLanguage=english"
url = url_template.format(game_id)

chrome_options = Options()
language = "en-US"
chrome_options.add_argument(f"--lang={language}")

driver = webdriver.Chrome(service=service, options=chrome_options)
driver.maximize_window() # Maximizing window to ensure more content loads initially
driver.get(url)

print(f"Successfully navigated to: {driver.title}")


def get_curr_scroll_pos(driver):
    """Returns the current scroll position of the page."""
    return driver.execute_script("return window.pageYOffset;")

def scroll_to_bottom(driver):
    """Scrolls the page to the very bottom."""
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")

def get_steam_id(card):
    """Extracts the Steam ID from a review card element."""
    profile_url = card.find_element(By.XPATH,'.//div[@class="apphub_friend_block"]//a').get_attribute('href')
    steam_id = profile_url.split('/')[-2]
    return steam_id

def scrape_review_data(card):
    """Extracts review text, length, thumb status, play hours, and date from a review card."""
    content_element = card.find_element(By.CLASS_NAME, 'apphub_CardTextContent')

    # Extract the date posted
    date_element = content_element.find_element(By.CLASS_NAME, 'date_posted')
    date_posted = date_element.text.strip()

    # Extract the full text content of the review block
    full_text = content_element.text.strip()

    # The review text is the full text minus the date string
    review_text = full_text.replace(date_posted, '').strip()
    review_length = len(review_text.replace(" ","")) # Calculate length excluding spaces

    # Extract the status (e.g., Recommended, Not Recommended)
    thumb_text = card.find_element(By.XPATH, './/div[@class="reviewInfo"]/div[@class="title"]').text.strip()

    # Extract play hours
    play_hours = card.find_element(By.XPATH, './/div[@class="reviewInfo"]/div[@class="hours"]').text.strip()

    return review_text, review_length, thumb_text, play_hours, date_posted

#Main Scraping Logic
reviews = []
steam_ids_set = set()
# Increased max_scroll_attempts significantly for more comprehensive scraping
max_scroll_attempts = 40

try:
    scroll_attempt = 0

    while scroll_attempt < max_scroll_attempts:
        
        prev_review_count = len(driver.find_elements(By.CLASS_NAME,'apphub_Card'))
        print(f"Current reviews before scroll: {prev_review_count}")

        scroll_to_bottom(driver)

        try:
            # Use WebDriverWait to explicitly wait for new review cards to appear.
            # It waits up to 10 seconds for the number of 'apphub_Card' elements to increase.
            # This is more robust than a fixed time.sleep().
            WebDriverWait(driver, 10).until(
                lambda d: len(d.find_elements(By.CLASS_NAME, 'apphub_Card')) > prev_review_count
            )
            print(f"New reviews detected after scroll.")
            scroll_attempt = 0
        except TimeoutException:
            scroll_attempt += 1
            print(f"No new reviews loaded within timeout. Attempt {scroll_attempt}/{max_scroll_attempts}")
            if scroll_attempt >= max_scroll_attempts:
                print("Reached max scroll attempts without new content. Stopping scraping.")
                break

        cards = driver.find_elements(By.CLASS_NAME,'apphub_Card')
        new_review_count = len(cards)
        print(f"Total reviews after scroll: {new_review_count}")

        # Iterate through the newly found review cards
        # We iterate through all 'cards' each time to ensure we catch any that might have been
        # missed or re-rendered, but the 'steam_ids_set' prevents duplicates.

        for card in cards:
            try:
                steam_id = get_steam_id(card)
            except NoSuchElementException:
                continue # Skip this card if the steam_id element is missing
            except Exception as e:
                print(f"Failed to get steam_id for a card: {e}")
                continue

            # Skip if this Steam ID has already been processed
            if steam_id in steam_ids_set:
                continue

            try:
                review_data = scrape_review_data(card)
                # Ensure the scraped data has the expected number of elements
                if len(review_data) == 5:
                    reviews.append(review_data)
                    steam_ids_set.add(steam_id)
                    print(f"Collected review #{len(reviews)} from {steam_id}")
                else:
                    print(f"Incomplete review data for {steam_id}. Skipping.")

                # Periodically save a backup of the collected data
                if len(reviews) % 100 == 0:
                    df = pd.DataFrame(reviews, columns=[
                        'Review Text', 'Review Length', 'Thumb Text', 'Play Hours', 'Date Posted'])
                    df.to_csv("steam_reviews_backup.csv", index=False)
                    print(f"Saved backup with {len(reviews)} reviews.")
            except NoSuchElementException as e:
                continue # Skip this card if an element is missing inside scrape_review_data
            except Exception as e:
                print(f"General error processing review from {steam_id}: {e}")

except Exception as e:
    print("An unexpected error occurred during the main scraping loop:", e)

finally:
    print(f"\nScraping finished. Total reviews collected: {len(reviews)}")
    if driver:
        driver.quit()

if reviews:
    final_df = pd.DataFrame(reviews, columns=[
        'Review Text', 'Review Length', 'Thumb Text', 'Play Hours', 'Date Posted'])
    final_df.to_csv("steam_reviews_final.csv", index=False)
    print("Final data saved to steam_reviews_final.csv")
else:
    print("No reviews were collected.")



In [ ]:
df = pd.read_csv("steam_reviews_final.csv")
df

,Review Text,Review Length,Thumb Text,Play Hours,Date Posted
0,10 out of 10 game but MJ is the main reason wh...,67,Recommended,44.8 hrs on record,Posted: 6 July
1,best game ever,12,Recommended,16.1 hrs on record,Posted: 6 July
2,"this game is very good, is an anventure game i...",43,Recommended,5.0 hrs on record,Posted: 6 July
3,not well optimized for now,22,Not Recommended,2.5 hrs on record,Posted: 6 July
4,cause,5,Recommended,7.2 hrs on record,Posted: 6 July
...,...,...,...,...,...
1995,"The story in this game is disappointing, Krave...",847,Not Recommended,27.7 hrs on record,Posted: April 9
1996,My favourite spider man game by far ^^\n\nI lo...,561,Recommended,41.0 hrs on record,Posted: April 9
1997,good game. just wished it wasnt so heavy on th...,57,Recommended,34.1 hrs on record,Posted: April 9
1998,Not as good as the first one but a great game ...,140,Recommended,26.0 hrs on record,Posted: April 9
